In [2]:
pip install transformers torch accelerate bitsandbytes


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/24.1 MB ? eta -:--:--Downloading pymupdf-1.26.7-cp310-abi3-manylinux_2_28_x86_64.whl (24.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 30.7 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 30.7 MB/s  0:00:00m0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [23]:
pip install dotenv

Note: you may need to restart the kernel to use updated packages.


In [24]:
from dotenv import load_dotenv
import os

load_dotenv()                 # memuat dari .env di direktori kerja
my_secret_key = os.getenv("HF_TOKEN")

In [25]:
import torch
from transformers import AutoTokenizer ,AutoModelForCausalLM, BitsAndBytesConfig, pipeline

In [26]:
# model_id="meta-llama/Llama-3.1-8B"
model_id="meta-llama/Llama-3.1-8B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=my_secret_key)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto", token=my_secret_key)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.27it/s]


In [27]:
# Membuat dataset untuk fine-tuning tentang Budi
import json

# Dataset tentang informasi personal Budi
budi_dataset = [
    {
        "input": "Siapa itu Budi?",
        "output": "Budi adalah seorang mahasiswa Teknik Informatika semester 6 di Universitas Gadjah Mada. Dia berusia 21 tahun dan berasal dari Yogyakarta. Budi memiliki passion yang besar di bidang artificial intelligence dan machine learning."
    },
    {
        "input": "Apa hobi Budi?",
        "output": "Hobi Budi sangat beragam. Dia suka coding terutama dengan Python dan JavaScript. Di waktu senggang, Budi gemar bermain game online, membaca buku tentang teknologi, dan menonton film sci-fi. Dia juga suka ngopi sambil ngoding di cafe-cafe di sekitar kampus."
    },
    {
        "input": "Bagaimana kepribadian Budi?",
        "output": "Budi adalah orang yang introvert tapi ramah. Dia sangat teliti dalam mengerjakan tugas dan selalu berusaha memberikan hasil terbaik. Budi juga dikenal sebagai problem solver yang handal di kelasnya. Meskipun pendiam, dia selalu siap membantu teman-temannya yang kesulitan dalam mata kuliah pemrograman."
    },
    {
        "input": "Apa makanan favorit Budi?",
        "output": "Makanan favorit Budi adalah nasi gudeg khas Yogyakarta dan mie ayam. Dia juga sangat suka minum kopi, terutama kopi tubruk dan americano. Kalau lagi stress ngerjain tugas, Budi biasanya ngemil keripik atau coklat sambil ngoding."
    },
    {
        "input": "Apa cita-cita Budi?",
        "output": "Cita-cita Budi adalah menjadi AI researcher atau data scientist di perusahaan teknologi besar seperti Google atau Microsoft. Dia bermimpi bisa mengembangkan teknologi AI yang bermanfaat untuk banyak orang, khususnya di bidang healthcare dan education."
    },
    {
        "input": "Bagaimana keluarga Budi?",
        "output": "Budi anak kedua dari tiga bersaudara. Ayahnya bekerja sebagai guru SMA dan ibunya seorang pegawai bank. Kakaknya sudah bekerja sebagai software engineer di Jakarta, sedangkan adiknya masih SMA kelas 2. Keluarga Budi sangat supportif dengan passion-nya di bidang teknologi."
    },
    {
        "input": "Apa prestasi Budi di kampus?",
        "output": "Budi memiliki IPK 3.85 dan pernah menjadi juara 2 lomba programming tingkat fakultas. Dia juga aktif sebagai asisten dosen untuk mata kuliah Algoritma dan Struktur Data. Tahun lalu, Budi berhasil lolos magang di startup teknologi lokal sebagai junior developer."
    },
    {
        "input": "Apa rencana Budi setelah lulus?",
        "output": "Setelah lulus, Budi berencana untuk langsung bekerja sambil mengambil program magister part-time. Dia sudah mulai apply ke beberapa perusahaan teknologi untuk posisi machine learning engineer atau data analyst. Budi juga mempertimbangkan untuk melanjutkan S2 di luar negeri jika ada kesempatan beasiswa."
    }
]

print(f"Dataset tentang Budi dibuat dengan {len(budi_dataset)} contoh data")
print("\nContoh data pertama:")
print(f"Input: {budi_dataset[0]['input']}")
print(f"Output: {budi_dataset[0]['output'][:100]}...")

Dataset tentang Budi dibuat dengan 8 contoh data

Contoh data pertama:
Input: Siapa itu Budi?
Output: Budi adalah seorang mahasiswa Teknik Informatika semester 6 di Universitas Gadjah Mada. Dia berusia ...


In [29]:
# Install library tambahan untuk fine-tuning
!pip install peft datasets trl

In [41]:
# Import library untuk fine-tuning
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer
from datasets import Dataset

In [42]:
# Konversi dataset ke format yang sesuai untuk training dengan labels
from datasets import Dataset
import pandas as pd

def format_and_tokenize(examples):
    texts = []
    for inp, out in zip(examples['input'], examples['output']):
        # Format sederhana dengan EOS token
        text = f"Pertanyaan: {inp}\nJawaban: {out}{tokenizer.eos_token}"
        texts.append(text)
    
    # Tokenisasi
    tokenized = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=256,
        return_tensors=None
    )
    
    # Set labels = input_ids untuk causal LM (PENTING!)
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

# Konversi ke Hugging Face Dataset
df = pd.DataFrame(budi_dataset)
hf_dataset = Dataset.from_pandas(df)

# Tokenisasi
tokenized_dataset = hf_dataset.map(
    format_and_tokenize,
    batched=True,
    remove_columns=hf_dataset.column_names
)

print(f"✅ Dataset berhasil diproses: {len(tokenized_dataset)} sampel")
print(f"Kolom: {tokenized_dataset.column_names}")

Map: 100%|██████████| 8/8 [00:00<00:00, 1927.53 examples/s]

✅ Dataset berhasil diproses: 8 sampel
Kolom: ['input_ids', 'attention_mask', 'labels']


In [43]:
# Persiapkan model untuk k-bit training
model = prepare_model_for_kbit_training(model)

# Konfigurasi LoRA - LEBIH AGRESIF
lora_config = LoraConfig(
    r=32,  # Rank lebih tinggi untuk kapasitas lebih besar
    lora_alpha=64,  # Alpha lebih tinggi
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  # Semua modul attention + FFN
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Terapkan LoRA ke model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/home/oppir/TA-OPH/25-26/TA-07/.venv/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/oppir/TA-OPH/25-26/TA-07/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 83,886,080 || all params: 8,114,147,328 || trainable%: 1.0338


In [44]:
# Konfigurasi training - LEBIH AGRESIF untuk dataset kecil
training_args = TrainingArguments(
    output_dir="./budi-llama-finetuned",
    num_train_epochs=20,  # BANYAK epoch untuk dataset kecil
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,  # Learning rate lebih tinggi
    fp16=True,
    save_steps=100,
    logging_steps=2,
    save_total_limit=2,
    warmup_steps=2,
    report_to="none",
    gradient_checkpointing=False,
    max_grad_norm=0.3,  # Gradient clipping
)

print("✅ Training arguments berhasil dikonfigurasi (AGGRESSIVE MODE)")

✅ Training arguments berhasil dikonfigurasi (AGGRESSIVE MODE)


In [45]:
# Data Collator untuk Causal LM (TANPA MLM!)
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # PENTING: False untuk Causal LM
)

# Buat Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("✅ Trainer berhasil dibuat. Siap untuk training!")

✅ Trainer berhasil dibuat. Siap untuk training!


In [46]:
# Mulai training
print("🚀 Memulai fine-tuning AGRESIF...")
print("=" * 60)
trainer.train()
print("=" * 60)
print("✅ Training selesai!\n")

# Test model LANGSUNG setelah training
print("=" * 60)
print("🧪 TESTING MODEL YANG SUDAH DI-FINE-TUNE")
print("=" * 60)

# Set model ke mode evaluasi
model.eval()

def test_model(question):
    # Format prompt yang SAMA dengan training
    prompt = f"Pertanyaan: {question}\nJawaban:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate dengan parameter yang lebih deterministik
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.3,  # Lebih rendah = lebih fokus pada training data
            top_p=0.85,
            do_sample=True,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    # Decode hanya bagian yang di-generate (skip prompt)
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Ekstrak hanya jawaban
    if "Jawaban:" in full_text:
        answer = full_text.split("Jawaban:", 1)[1].strip()
    else:
        answer = full_text[len(prompt):].strip()
    
    return answer

# Test dengan pertanyaan dari dataset
test_questions = [
    "Siapa itu Budi?",
    "Apa hobi Budi?",
    "Apa makanan favorit Budi?",
    "Bagaimana kepribadian Budi?",
    "Apa cita-cita Budi?",
    "Bagaimana keluarga Budi?",
    "Apa prestasi Budi di kampus?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n[{i}/{len(test_questions)}] 📝 {question}")
    print("💬 Jawaban:")
    response = test_model(question)
    print(response)
    print("-" * 60)

🚀 Memulai fine-tuning AGRESIF...


/home/oppir/TA-OPH/25-26/TA-07/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
2,1.794100
4,0.959100
6,0.428000
8,0.235100
10,0.162600
12,0.090700
14,0.056200
16,0.047700
18,0.052200
20,0.050300


✅ Training selesai!

🧪 TESTING MODEL YANG SUDAH DI-FINE-TUNE

[1/7] 📝 Siapa itu Budi?
💬 Jawaban:
Budi adalah seorang mahasiswa Teknik Informatika semester 6 di Universitas Gadjah Mada. Dia berusia 21 tahun dan berasal dari Yogyakarta. Budi memiliki passion yang besar di bidang artificial intelligence dan machine learning. Dia sudah mulai apply knowledge tersebut di perusahaan teknologi untuk mengembangkan solusi AI yang bermanfaat untuk banyak orang. Budi juga aktif sebagai asisten dosen untuk mata kuliah Algoritma dan Struktur Data. Tahun lalu, Budi berhasil lolos magang di startup lokal sebagai junior developer. Peranannya meliputi membantu desain sistem dan integrasi teknologi baru. Meskipun masih SMA kelas 2
------------------------------------------------------------

[2/7] 📝 Apa hobi Budi?
💬 Jawaban:
Budi adalah seorang mahasiswa Teknik Informatika semester 6 di Universitas Gadjah Mada. Dia berusia 21 tahun dan berasal dari Yogyakarta. Budi memiliki passion yang besar di bidang a

In [ ]:
# Simpan model yang sudah di-fine-tune (OPSIONAL)
model.save_pretrained("./budi-llama-final-model")
tokenizer.save_pretrained("./budi-llama-final-model")
print("Model berhasil disimpan di folder './budi-llama-final-model'")